# Web Export (3D Tiles & Potree)

Export point clouds for browser-based 3D viewers like CesiumJS and Potree.

**Modules used:** `occulus.export`

In [ ]:
import numpy as np
from pathlib import Path
import tempfile

from occulus.types import PointCloud
from occulus.export import export_3dtiles, export_potree

## Create a sample point cloud

In [ ]:
rng = np.random.default_rng(42)
n = 50_000
xyz = np.column_stack([
    rng.uniform(0, 500, n),
    rng.uniform(0, 500, n),
    rng.exponential(5, n) + rng.uniform(0, 2, n),
])
cloud = PointCloud(xyz)
print(f"Points: {cloud.n_points:,}")

## Export as 3D Tiles (CesiumJS)

Produces a `tileset.json` and `.pnts` binary files.

In [ ]:
out_3d = Path(tempfile.mkdtemp()) / "cesium"
tileset_path = export_3dtiles(cloud, out_3d, max_points_per_tile=10_000)

print(f"Tileset: {tileset_path}")
print(f"Files created:")
for f in sorted(out_3d.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

## Export as Potree

Produces an octree hierarchy with `metadata.json` and `.bin` chunks.

In [ ]:
out_potree = Path(tempfile.mkdtemp()) / "potree"
meta_path = export_potree(cloud, out_potree, max_depth=8)

print(f"Metadata: {meta_path}")

import json
meta = json.loads(meta_path.read_text())
print(f"Total points: {meta['points']:,}")
print(f"Octree nodes: {len(meta['hierarchy'])}")
print(f"Bounding box: {meta['boundingBox']}")